# Lab 10 - Turn tool calls into a regression test

## What are we testing?

Lab 9 evaluated the full path through an agent. This lab focuses on one smaller contract: for a known user request, did the agent choose the expected tool and pass the expected arguments?

A tool call has two business-relevant parts:

```text
tool name + arguments
```

For example, choosing `get_ipc_assessment` is not enough if the user asked about General Medicine (`4B`) and the call sends Surgical (`2A`).

A **regression test** keeps examples of correct and incorrect behavior and reruns them after changes. It helps detect when a prompt, model or tool description breaks behavior that previously worked.

This lab uses two complementary checks:

| Check | What it proves or explains |
|---|---|
| Exact local comparison | The tool name and complete arguments match the expected call |
| Tool Selection | The chosen tool was appropriate and necessary for the request |
| Tool Call Accuracy | The overall tool choice and inputs made semantic sense |

The expected call is the test's **oracle**. Before exact comparison, the code removes transport-only details such as `tool_call_id`; this is called **normalization**.

You will test four frozen cases: correct read, wrong ward argument, approved write, and a question that should make no tool call. Then you will repair the wrong argument and compare baseline and fixed runs.

## Before you start

Run `az login`, select Python 3.11+, and use the same Foundry project and model deployment as Lab 9. Cloud evaluator scores can vary.

These cases do not execute tools or test the booking backend. They test the agent's proposed call contract only.

Replace each `...` blank before running its cell.

In [ ]:
%pip install -q "azure-ai-projects==2.3.0" "azure-identity==1.25.3" "openai==2.54.0"

## 0. Connect and prepare evaluation runs

The next cell signs in, creates the Foundry evaluation client and defines the same asynchronous run helper used in Labs 8 and 9.

Foundry returns a run before cloud scoring is finished. The helper polls until the run reaches a final status, then waits until every submitted row has a result. It also converts SDK objects into ordinary Python data for printing.

**You should see** `Ready` and a unique suffix for the evaluation and run names.

In [ ]:
import copy
import os
import sys
import time
from uuid import uuid4

from azure.ai.projects import AIProjectClient
from azure.identity import AzureCliCredential

PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT", "")
MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "")
if sys.version_info < (3, 11):
    raise RuntimeError("Select a Python 3.11 or later kernel.")
if not PROJECT_ENDPOINT or not MODEL_DEPLOYMENT:
    raise ValueError("Set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME.")


def check_todos(**answers: object) -> None:
    still_open = [name for name, value in answers.items() if value is ...]
    if still_open:
        raise ValueError(f"Fill in these blanks first: {', '.join(still_open)}")


def primitive(value):
    return value.model_dump(mode="json") if hasattr(value, "model_dump") else value


def wait_for_run(eval_id, run, expected_items):
    deadline = time.monotonic() + 1200
    while run.status not in ("completed", "failed", "canceled"):
        if time.monotonic() > deadline:
            raise TimeoutError("Evaluation exceeded 20 minutes.")
        time.sleep(5)
        run = client.evals.runs.retrieve(run_id=run.id, eval_id=eval_id)
        print("status:", run.status)
    items = list(client.evals.runs.output_items.list(run_id=run.id, eval_id=eval_id))
    if run.status != "completed":
        raise RuntimeError(f"Evaluation ended as {run.status}: {primitive(getattr(run, 'error', None))}")
    item_deadline = time.monotonic() + 120
    while len(items) < expected_items:
        if time.monotonic() > item_deadline:
            raise TimeoutError(f"Only {len(items)}/{expected_items} output items became visible.")
        time.sleep(2)
        items = list(client.evals.runs.output_items.list(run_id=run.id, eval_id=eval_id))
    return run, items


SUFFIX = uuid4().hex[:8]
credential = AzureCliCredential()
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
client = project.get_openai_client(timeout=300, max_retries=0)
print(f"Ready. Suffix: {SUFFIX}")

## 1. Define strict tool contracts

The evaluator needs the same tool information the agent used when deciding what to call:

- the tool name identifies the action;
- the description explains when the action is appropriate;
- the parameter schema defines valid arguments.

The ward argument uses an enum, so only `4B`, `2A` or `ICU` is valid. `additionalProperties: false` rejects undeclared arguments. These strict schemas make malformed calls easy to detect and reduce ambiguity for semantic evaluation.

The definitions describe the read and write tools from Lab 6. They do not execute either function.

**You should see** the two tool names and confirmation that both schemas reject extra arguments.

In [ ]:
TOOL_DEFINITIONS = [
    {
        "name": "get_ipc_assessment",
        "description": "Look up a synthetic ward's current IPC self-assessment. General Medicine is Ward 4B, Surgical is 2A, and Intensive Care is ICU.",
        "parameters": {
            "type": "object",
            "properties": {"ward": {"type": "string", "enum": ["4B", "2A", "ICU"]}},
            "required": ["ward"],
            "additionalProperties": False,
        },
    },
    {
        "name": "schedule_ipc_review",
        "description": "Book a synthetic IPC review only after explicit human approval.",
        "parameters": {
            "type": "object",
            "properties": {
                "ward": {"type": "string", "enum": ["4B", "2A", "ICU"]},
                "component": {"type": "string"},
                "reason": {"type": "string"},
            },
            "required": ["ward", "component", "reason"],
            "additionalProperties": False,
        },
    },
]
assert all(tool["parameters"]["additionalProperties"] is False for tool in TOOL_DEFINITIONS)
print("Tools:", [tool["name"] for tool in TOOL_DEFINITIONS])

## 2. Create exact regression cases

Each row contains a user query, the tool calls produced by the agent and the calls expected by the test author.

| Case | Expected behavior |
|---|---|
| `T-01` | Read Ward `4B` with the correct argument |
| `T-02` | Fail: the right read tool receives `2A` instead of General Medicine's `4B` |
| `T-03` | Schedule the approved Ward `4B` review with all required arguments |
| `T-04` | Make no call for a conceptual question |

A **no-tool case** matters because unnecessary calls add latency and cost, and write tools can create side effects.

The generated calls contain `type` and `tool_call_id`, but those values only transport the call through the API. The expected calls keep the behavior that matters: tool name and arguments.

These are frozen call records. No function runs in this section.

**You should see** four unique case IDs and only `T-02` marked as the expected local failure.

In [ ]:
CASES = [
    {
        "case_id": "T-01",
        "query": "Look up Ward 4B's IPC assessment.",
        "tool_calls": [{"type": "tool_call", "tool_call_id": "t1", "name": "get_ipc_assessment", "arguments": {"ward": "4B"}}],
        "expected_calls": [{"name": "get_ipc_assessment", "arguments": {"ward": "4B"}}],
        "tool_definitions": TOOL_DEFINITIONS,
        "expected_local": "pass",
    },
    {
        "case_id": "T-02",
        "query": "How is the General Medicine ward performing on IPC?",
        "tool_calls": [{"type": "tool_call", "tool_call_id": "t2", "name": "get_ipc_assessment", "arguments": {"ward": "2A"}}],
        "expected_calls": [{"name": "get_ipc_assessment", "arguments": {"ward": "4B"}}],
        "tool_definitions": TOOL_DEFINITIONS,
        "expected_local": "fail",
    },
    {
        "case_id": "T-03",
        "query": "Human approval is granted. Book Ward 4B's workload and staffing review because its score is 1/5.",
        "tool_calls": [{"type": "tool_call", "tool_call_id": "t3", "name": "schedule_ipc_review", "arguments": {"ward": "4B", "component": "workload, staffing and bed occupancy", "reason": "Score is 1/5"}}],
        "expected_calls": [{"name": "schedule_ipc_review", "arguments": {"ward": "4B", "component": "workload, staffing and bed occupancy", "reason": "Score is 1/5"}}],
        "tool_definitions": TOOL_DEFINITIONS,
        "expected_local": "pass",
    },
    {
        "case_id": "T-04",
        "query": "In one sentence, what is a function tool?",
        "tool_calls": [],
        "expected_calls": [],
        "tool_definitions": TOOL_DEFINITIONS,
        "expected_local": "pass",
    },
]
assert len({row["case_id"] for row in CASES}) == len(CASES)
assert [row["case_id"] for row in CASES if row["expected_local"] == "fail"] == ["T-02"]
print("Cases:", [row["case_id"] for row in CASES])

### To-Do 1 - Compare only the call contract

An exact test should ignore values that do not change behavior while preserving everything the interface depends on.

Set `COMPARISON_FIELDS` to:

- `name`, which identifies the function;
- `arguments`, which contains the complete input sent to it.

The normalizer removes `type` and `tool_call_id`, then compares the ordered calls with `expected_calls`. Order remains significant because a multi-step agent may need to read before it writes.

**Predict:** if you compared only `name`, why would `T-02` incorrectly pass?

**Key concept:** normalization removes transport noise; it must not weaken the business contract.

<details><summary>Show solution code</summary>

```python
COMPARISON_FIELDS = ("name", "arguments")
```

</details>

In [ ]:
COMPARISON_FIELDS = ...  # TODO 1: fields that define the call contract.
check_todos(COMPARISON_FIELDS=COMPARISON_FIELDS)


def normalize_calls(calls):
    return [
        {field: call.get(field, {} if field == "arguments" else None) for field in COMPARISON_FIELDS}
        for call in calls
    ]


def exact_result(row):
    return "pass" if normalize_calls(row["tool_calls"]) == row["expected_calls"] else "fail"


BASELINE_RESULTS = {row["case_id"]: exact_result(row) for row in CASES}
assert BASELINE_RESULTS == {row["case_id"]: row["expected_local"] for row in CASES}
print("PASS - exact comparison found the wrong ward argument:", BASELINE_RESULTS)

## 3. Add semantic tool-call evaluators

The exact comparison tells you whether a call matches the oracle. Foundry's model-based evaluators ask broader questions:

- **Tool Selection:** was this function appropriate and necessary for the user's request?
- **Tool Call Accuracy:** were the selected tool and its arguments reasonable for the request and tool definition?

This distinction explains a useful disagreement: `T-02` chose the correct read function, so Tool Selection may pass, while its wrong ward argument should fail the exact comparison and may fail Tool Call Accuracy.

The cloud run includes only `T-01`, `T-02` and `T-03` because this evaluator input shape currently requires at least one tool call. `T-04` remains protected by the local exact suite.

The next cell creates one evaluation, scores the three call-bearing baseline rows and prints each evaluator's score, label and reason.

**You should see** three cloud results and a Foundry report URL.

In [ ]:
from azure.ai.projects.models import TestingCriterionAzureAIEvaluator
from openai.types.eval_create_params import DataSourceConfigCustom

data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "case_id": {"type": "string"},
            "query": {"type": "string"},
            "tool_calls": {"type": "array"},
            "expected_calls": {"type": "array"},
            "tool_definitions": {"type": "array"},
            "expected_local": {"type": "string"},
        },
        "required": ["case_id", "query", "tool_calls", "tool_definitions"],
        "additionalProperties": False,
    },
    include_sample_schema=True,
)
common_mapping = {
    "query": "{{item.query}}",
    "tool_calls": "{{item.tool_calls}}",
    "tool_definitions": "{{item.tool_definitions}}",
}
testing_criteria = [
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="tool_call_accuracy",
        evaluator_name="builtin.tool_call_accuracy",
        initialization_parameters={"deployment_name": MODEL_DEPLOYMENT},
        data_mapping=common_mapping,
    ),
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="tool_selection",
        evaluator_name="builtin.tool_selection",
        initialization_parameters={"deployment_name": MODEL_DEPLOYMENT},
        data_mapping=common_mapping,
    ),
]
CLOUD_CASES = [row for row in CASES if row["tool_calls"]]
assert [row["case_id"] for row in CLOUD_CASES] == ["T-01", "T-02", "T-03"]

evaluation = client.evals.create(
    name=f"day2-tool-regression-eval-{SUFFIX}",
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,
)
baseline_run = client.evals.runs.create(
    eval_id=evaluation.id,
    name=f"day2-tool-regression-baseline-{SUFFIX}",
    metadata={"suite": "ipc-tool-calls-v1"},
    data_source={
        "type": "jsonl",
        "source": {"type": "file_content", "content": [{"item": row} for row in CLOUD_CASES]},
    },
)
baseline_run, baseline_items = wait_for_run(evaluation.id, baseline_run, len(CLOUD_CASES))
for item in baseline_items:
    data = primitive(item)
    print("\n", data.get("datasource_item", {}).get("case_id", data.get("item_id")))
    for result in data.get("results", []):
        print(f"  {result.get('name')}: score={result.get('score')} label={result.get('label')}")
        print(f"    {result.get('reason')}")
print({"report_url": getattr(baseline_run, "report_url", None)})

### To-Do 2 - Repair the wrong ward argument

Keep the failing baseline and make a corrected copy. This preserves proof that the suite catches the original defect.

`T-02` already selects the correct read tool, but it sends the Surgical ward identifier. Set `GENERAL_MEDICINE_WARD` to the identifier declared in the tool description.

The code changes only that argument in the copied suite, verifies all four exact cases now pass, confirms the original row is still wrong, and confirms the no-tool case is still present. It then submits the three call-bearing fixed rows as a second cloud run.

**Predict:** why might Tool Selection already pass before the repair while Tool Call Accuracy and the exact test fail?

**Key concept:** tool choice and tool inputs are separate failure surfaces.

<details><summary>Show solution code</summary>

```python
GENERAL_MEDICINE_WARD = "4B"
```

</details>

In [ ]:
GENERAL_MEDICINE_WARD = ...  # TODO 2: the ward identifier in the tool contract.
check_todos(GENERAL_MEDICINE_WARD=GENERAL_MEDICINE_WARD)

FIXED_CASES = copy.deepcopy(CASES)
fixed_t02 = next(row for row in FIXED_CASES if row["case_id"] == "T-02")
fixed_t02["tool_calls"][0]["arguments"]["ward"] = GENERAL_MEDICINE_WARD
fixed_t02["expected_local"] = "pass"
assert all(exact_result(row) == "pass" for row in FIXED_CASES)
assert next(row for row in CASES if row["case_id"] == "T-02")["tool_calls"][0]["arguments"]["ward"] == "2A"
assert next(row for row in FIXED_CASES if row["case_id"] == "T-04")["tool_calls"] == []

fixed_cloud_cases = [row for row in FIXED_CASES if row["tool_calls"]]
fixed_run = client.evals.runs.create(
    eval_id=evaluation.id,
    name=f"day2-tool-regression-fixed-{SUFFIX}",
    metadata={"suite": "ipc-tool-calls-v1", "variant": "ward-argument-fix"},
    data_source={
        "type": "jsonl",
        "source": {
            "type": "file_content",
            "content": [{"item": row} for row in fixed_cloud_cases],
        },
    },
)
fixed_run, fixed_items = wait_for_run(evaluation.id, fixed_run, len(fixed_cloud_cases))
print({"baseline_report": getattr(baseline_run, "report_url", None)})
print({"fixed_report": getattr(fixed_run, "report_url", None)})

## Verify the repaired regression suite

The exact local suite is the release gate for this known call contract. The model-based scores are supporting evidence, so the final check verifies that results exist rather than requiring one exact score.

It asserts that:

- only `T-02` fails in the original suite;
- all four copied cases pass after the ward repair;
- both cloud runs completed with three scored rows;
- the no-tool case remains in the four-row local suite;
- every submitted cloud row returned evaluator results.

**You should see** a `PASS` message confirming the exact repair and both semantic comparison runs.

In [ ]:
assert BASELINE_RESULTS == {"T-01": "pass", "T-02": "fail", "T-03": "pass", "T-04": "pass"}
assert all(exact_result(row) == "pass" for row in FIXED_CASES)
assert baseline_run.status == fixed_run.status == "completed"
assert len(baseline_items) == len(fixed_items) == 3
assert len(FIXED_CASES) == 4 and FIXED_CASES[-1]["tool_calls"] == []
assert all(primitive(item).get("results") for item in baseline_items + fixed_items)
print("PASS - all four exact cases pass after repair and both semantic comparison runs are available.")

## Interpret exact and semantic results together

The two layers can disagree because they answer different questions:

| Exact test | Semantic evaluator | What to inspect |
|---|---|---|
| Fail | Fail | Likely regression: inspect the generated call and expected oracle |
| Fail | Pass | The judge may accept an equivalent call, or the oracle may intentionally be stricter |
| Pass | Fail | The exact oracle may be incomplete even though the values match |
| Pass | Pass | Good evidence for this case, not proof for unseen requests |

Do not automatically weaken an exact rule because a model-based evaluator passes it. First decide whether the rule protects a real API contract, authorization boundary or data invariant.

## What you learned

- A tool-call regression test compares known requests with expected tool names and arguments.
- Normalization removes transport details without discarding business-relevant fields.
- Tool Selection evaluates whether the function choice made sense.
- Tool Call Accuracy evaluates the choice and inputs together.
- A no-tool expectation is a real regression case and remains in the local suite.
- Preserving a failing baseline proves that the test can detect the defect it was designed for.
- Tool-call tests inspect proposed calls; they do not execute tools or test backend reliability.

**Check your understanding**

1. The right function receives the wrong ward. Which checks should fail?
2. The exact suite passes, but the agent makes two unnecessary reads. What kind of check is missing?
3. Why must `T-04` stay in the local suite even though it is excluded from the cloud run?

<details><summary>Compare your answers</summary>

1. The exact test and Tool Call Accuracy should fail; Tool Selection may pass because the function choice is correct.
2. You need a semantic check of tool necessity or process efficiency.
3. It protects the requirement that a conceptual question should not trigger a tool, avoiding unnecessary cost, latency or side effects.

</details>

Further reading: [agent and tool evaluators](https://learn.microsoft.com/azure/foundry/concepts/evaluation-evaluators/agent-evaluators), [evaluation dataset schema](https://learn.microsoft.com/azure/foundry/observability/how-to/evaluation-dataset-schema), and [cloud evaluation](https://learn.microsoft.com/azure/foundry/observability/how-to/cloud-evaluation).

**Expected artifact:** all four exact cases pass after repair, with baseline and fixed Foundry report URLs for review.

**Finish:** the final cell closes local clients. The evaluation and both run reports remain in Foundry.

**Next:** Lab 11 tests safety behavior with deterministic abuse cases and optional cloud red teaming.

In [ ]:
client.close()
project.close()
credential.close()
print("Closed local clients. The evaluation and both run reports remain in Foundry.")